Notebook provided by Arto Javanainen, University of Jyvaskyla,
FINLAND. https://www.researchgate.net/profile/Arto_Javanainen

# Regular SRIM calculation: stopping power with the SR module

SRIM is a group of programs which calculate the stopping and range of ions (up to 2 GeV/amu)
into matter using a quantum mechanical treatment of ion-atom collisions. The ion has long
range interactions creating electron excitations and plasmons within the target, and its
charge state is described by an effective charge that depends on its velocity.

The **SR module** needs no ion histories, so it is fast. Set up SRIM once with `pysrim2 install` and check it with `pysrim2 doctor`
(see the [installation guide](https://pysrim2.readthedocs.io/en/latest/installation.html)). The notebook then finds SRIM
by itself, natively or in the container; no path is needed. SRIM-2013 only, **not** 2018.
SRIM is (c) SRIM.com (James F. Ziegler), free for non-commercial use only; see `LEGAL.rtf` next to your SRIM copy.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from srim import SR, Ion, Layer

In [ ]:
ion = Ion("Xe", energy=1.2e9)  # eV, the top of the table

layer = Layer(
    {
        "Si": {"stoich": 0.5, "E_d": 35.0, "lattice": 0.0, "surface": 3.0},
        "C": {"stoich": 0.5, "E_d": 20.0, "lattice": 0.0, "surface": 3.0},
    },
    density=3.21,
    width=10000.0,
)

sr = SR(layer, ion, output_type=5)  # 5 = MeV/(mg/cm2); 1 = eV/Angstrom
# If casting to floats fails in SRIM's output (e.g. 999,999), set sr.settings.energy_min = 1200
results = sr.run()  # uses the SRIM installed by `pysrim2 install`

In [ ]:
table = pd.DataFrame(
    results.data.T,
    columns=[
        "energy_keV",
        "electronic_stopping",
        "nuclear_stopping",
        "projected_range_um",
        "longitudinal_straggling_um",
        "lateral_straggling_um",
    ],
)
table.attrs["units"] = results.units
print(results.ion["name"], "in", list(results.target.values())[-1], "| stopping units:", results.units)
table.head()

In [ ]:
x = 1e-3 * table["energy_keV"] / results.ion["A1"]  # MeV per nucleon

fig, ax = plt.subplots()
ax.plot(x, table["electronic_stopping"], label="Se")
ax.plot(x, table["nuclear_stopping"], label="Sn")
ax.set(xlabel="MeV/u", ylabel=results.units, xscale="log", xlim=(1e-2, 1e2), ylim=(1, None))
ax.set_title(results.ion["name"] + " in SiC")
ax.legend()
ax.grid()
plt.show()